In [4]:
# %% [markdown]
# # Encoder-Decoder Modeli ile Basit Seq2Seq Görevi
# 
# Bu notebook'ta:
# 
# 1. **Encoder**: Girdi dizisini işleyip gizli durum (hidden state) elde eder.
# 2. **Decoder**: Encoder’dan gelen gizli durumu kullanarak adım adım çıktı (örneğin ters çevrilmiş dizi) üretir.
# 3. **Seq2Seq Wrapper**: Encoder ve Decoder'ı bir araya getirerek modelin eğitim sırasında nasıl çalıştığını gösterir.
#
# Örnek senaryo olarak, bir sayı dizisini tersine çevirecek basit bir sequence-to-sequence (seq2seq) görevi kullanacağız.
#
# **Not:** Bu örnekte dikkat dağıtıcı unsurlar (örneğin attention) eklenmemiş, modelin temel konseptlerine odaklanılmıştır.

# %% [code]
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Cihaz belirleme (GPU kullanabiliyorsanız onu tercih eder)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# %% [markdown]
# ## Encoder Sınıfı
# 
# Encoder, verilen girdi dizisini alır ve RNN (burada GRU) kullanarak bir gizli durum (hidden state) oluşturur.

# %% [code]
class EncoderRNN(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super(EncoderRNN, self).__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
    
    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.embedding(src)   # [src_len, batch_size, emb_dim]
        outputs, hidden = self.rnn(embedded)  # hidden: [1, batch_size, hidden_dim]
        return hidden

# %% [markdown]
# ## Decoder Sınıfı
# 
# Decoder, encoder tarafından oluşturulan gizli durumu alır ve adım adım çıktıyı üretir.  
# Her adımda, önceki çıktıyı veya "teacher forcing" ile gerçek hedefi kullanarak bir sonraki adımın tahminini yapar.

# %% [code]
class DecoderRNN(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super(DecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, input, hidden):
        # input: [batch_size] -> tek zaman adımı için token
        input = input.unsqueeze(0)        # [1, batch_size]
        embedded = self.embedding(input)  # [1, batch_size, emb_dim]
        output, hidden = self.rnn(embedded, hidden)
        # output: [1, batch_size, hidden_dim] -> fc_out uygulayarak tahmin üretiyoruz.
        prediction = self.fc_out(output.squeeze(0))  # [batch_size, output_dim]
        return prediction, hidden

# %% [markdown]
# ## Seq2Seq Wrapper Sınıfı
# 
# Bu sınıf encoder ve decoder'ı birleştirir.  
# - Önce encoder, girdi dizisini gizli durum vektörüne çevirir.  
# - Daha sonra decoder, bu gizli durumu ve başlangıç token'ını kullanarak çıktı dizisini üretir.
#
# Ayrıca `teacher forcing` mekanizması kullanılarak, eğitim sırasında modelin hızlı öğrenmesi sağlanır.

# %% [code]
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: [src_len, batch_size]
        # trg: [trg_len, batch_size]
        trg_len = trg.shape[0]
        batch_size = trg.shape[1]
        output_dim = self.decoder.fc_out.out_features
        
        # Tüm zaman adımlarının tahminlerini saklamak için
        outputs = torch.zeros(trg_len, batch_size, output_dim).to(self.device)
        
        # Encoder'ın çıktısı: son gizli durum
        hidden = self.encoder(src)
        
        # İlk giriş token'ı: başlangıç (örneğin <sos>)
        input = trg[0, :]  # [batch_size]
        
        for t in range(1, trg_len):
            # Her adımda decoder çağırılır
            output, hidden = self.decoder(input, hidden)
            outputs[t] = output
            # Tahmin edilen token
            top1 = output.argmax(1)
            # Teacher forcing: Rastgele bir olasılıkla gerçek hedef token kullanılır
            input = trg[t] if random.random() < teacher_forcing_ratio else top1
        
        return outputs

# %% [markdown]
# ## Basit Bir Eğitim Örneği: Sayı Dizilerini Ters Çevirmek
#
# Bu bölümde, modelin sıralı veriler üzerinde nasıl eğitileceğini göstereceğiz.
#
# **Veri Seti:**  
# Örneğimizde, her örnek girdi dizisinin elemanlarını tersine çevirip aynı çıktıyı vermesini hedefleyeceğiz.
#
# Örneğin,  
# **Girdi:** `[1, 2, 3, 4]`  
# **Hedef:** `[<sos>, 4, 3, 2, 1, <eos>]`
#
# Basitlik açısından şu varsayımlarda bulunacağız:
# - Giriş ve çıkış token'ları aynı sayı aralığından alınacak.
# - `<sos>` (başlangıç) token'ı 0, `<eos>` (bitiş) token'ı ise max token+1 olacak.

# %% [code]
# Hiperdogan parametreler
INPUT_DIM = 12  # 0: <sos>, 1-10: gerçek token'lar, 11: <eos>
OUTPUT_DIM = INPUT_DIM  # Çıkış da aynı token uzayı
EMB_DIM = 8
HIDDEN_DIM = 16

# Model tanımlamaları
encoder = EncoderRNN(INPUT_DIM, EMB_DIM, HIDDEN_DIM).to(device)
decoder = DecoderRNN(OUTPUT_DIM, EMB_DIM, HIDDEN_DIM).to(device)
model = Seq2Seq(encoder, decoder, device).to(device)

# %% [markdown]
# ### Eğitim için Örnek Veri Oluşturma
#
# Basit veri seti üretelim. Her örnekte bir sayı dizisi rastgele oluşturulacak, ve hedef dizi bu dizinin tersine çevrilmiş hali olacak.
#
# **Not:** İlk token hedef dizide `<sos>` olarak 0, son token `<eos>` olarak 11 olacak.

# %% [code]
def generate_example(min_len=3, max_len=6):
    seq_len = random.randint(min_len, max_len)
    # Gerçek sayılar 1-10 arasında olacak.
    seq = [random.randint(1, 10) for _ in range(seq_len)]
    # Girdi: sıradan dizi, çıktıda <sos> eklenip, sonrasında ters çevrilmiş dizi ve <eos> eklenir.
    src = seq  # giriş dizisi
    trg = [0] + list(reversed(seq)) + [11]  # <sos> = 0, <eos> = 11
    return src, trg

# Test edelim
example_src, example_trg = generate_example()
print("Girdi:", example_src)
print("Hedef:", example_trg)

# %% [markdown]
# ### Veri Dönüşümleri
#
# Modelimiz PyTorch tensorleri beklediğinden, dizileri tensorlere dönüştürüyoruz.
#
# Bizde diziler boyutunun `[seq_len, batch_size]` şeklinde olmasına dikkat edeceğiz.
# Burada batch_size için örnek olarak 1 kullanalım.

# %% [code]
def tensorify(seq, max_len=None):
    # Eğer max_len verilmezse dizinin kendisini kullanırız
    if max_len is None:
        max_len = len(seq)
    # Padding'e gerek duymuyorsak direkt tensor
    tensor = torch.LongTensor(seq)
    return tensor

# Tek bir örnek üzerinde çalışıyoruz:
src_tensor = tensorify(example_src).unsqueeze(1).to(device)  # [src_len, 1]
trg_tensor = tensorify(example_trg).unsqueeze(1).to(device)  # [trg_len, 1]
print("Girdi tensor shape:", src_tensor.shape)
print("Hedef tensor shape:", trg_tensor.shape)

# %% [markdown]
# ## Eğitim Döngüsü
#
# Basit bir eğitim döngüsü oluşturalım.  
# – Modelin çıktısını, hedef ile karşılaştırarak kayıp hesaplarız.
# – Burada CrossEntropyLoss kullanıyoruz.  
# – Arka yayılım (backpropagation) ile parametreleri güncelliyoruz.

# %% [code]
optimizer = optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

def train(model, iterator, optimizer, criterion, clip=1):
    model.train()
    epoch_loss = 0
    
    for src_tensor, trg_tensor in iterator:
        optimizer.zero_grad()
        output = model(src_tensor, trg_tensor)
        # output: [trg_len, batch_size, output_dim]
        # Hedef: [trg_len, batch_size] -> ilk token <sos> olduğundan kayıp hesaplanmaz.
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg_tensor[1:].view(-1)
        
        loss = criterion(output, trg)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        epoch_loss += loss.item()
    
    return epoch_loss / len(iterator)

# %% [markdown]
# ### Eğitim için Veri Üretimi
#
# Basitçe, örneklerden oluşan bir "dataloader" oluşturalım.

# %% [code]
def generate_data(num_samples):
    data = []
    for _ in range(num_samples):
        src_seq, trg_seq = generate_example()
        # Tensorlere çeviriyoruz
        src_tensor = tensorify(src_seq).unsqueeze(1)  # [src_len, 1]
        trg_tensor = tensorify(trg_seq).unsqueeze(1)   # [trg_len, 1]
        data.append((src_tensor.to(device), trg_tensor.to(device)))
    return data

# 100 örnek oluşturalım
train_data = generate_data(100)

# %% [markdown]
# ### Model Eğitimi
#
# Şimdi modelimizi birkaç epoch boyunca eğitelim.

# %% [code]
NUM_EPOCHS = 10
for epoch in range(NUM_EPOCHS):
    loss = train(model, train_data, optimizer, criterion)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {loss:.4f}")

# %% [markdown]
# ## Model Testi
#
# Eğitim tamamlandıktan sonra, modeli test edelim.  
# Örnek olarak bir sayı dizisinin ters çevrilmesi nasıl gerçekleşiyor görelim.

# %% [code]
model.eval()
with torch.no_grad():
    test_src, test_trg = generate_example()
    src_tensor = tensorify(test_src).unsqueeze(1).to(device)
    trg_tensor = tensorify(test_trg).unsqueeze(1).to(device)
    
    output = model(src_tensor, trg_tensor, teacher_forcing_ratio=0)  # Teacher forcing kapalı
    # Tahmin edilen tokenları elde edelim.
    output_tokens = output.argmax(2).squeeze(1).tolist()
    
    print("Test Girdi:", test_src)
    print("Hedef (Ters):", test_trg)
    print("Model Tahmini:", output_tokens)
    
# %% [markdown]
# # Özet
#
# Bu notebook'ta:
# - PyTorch ile basit bir encoder (GRU tabanlı) ve decoder (GRU tabanlı) modeli tanımladık.
# - Bu iki yapıyı birleştirerek bir Seq2Seq modelini oluşturduk.
# - Basit bir sequence-to-sequence görevi için (sayı dizisini ters çevirme) eğitim ve test süreçlerini uyguladık.
#
# Böylece encoder–decoder mimarisi temellerini öğrenmiş oldunuz. İsterseniz bu temel yapı üzerine attention mekanizması veya Transformer tabanlı geliştirmeleri ekleyerek ilerleyebilirsiniz. 

Girdi: [5, 3, 5]
Hedef: [0, 5, 3, 5, 11]
Girdi tensor shape: torch.Size([3, 1])
Hedef tensor shape: torch.Size([5, 1])
Epoch 1/10, Loss: 2.4578
Epoch 2/10, Loss: 2.2969
Epoch 3/10, Loss: 2.1699
Epoch 4/10, Loss: 2.0734
Epoch 5/10, Loss: 1.9876
Epoch 6/10, Loss: 1.9043
Epoch 7/10, Loss: 1.8300
Epoch 8/10, Loss: 1.7812
Epoch 9/10, Loss: 1.7347
Epoch 10/10, Loss: 1.7017
Test Girdi: [9, 10, 8, 9, 2, 5]
Hedef (Ters): [0, 5, 2, 9, 8, 10, 9, 11]
Model Tahmini: [0, 5, 5, 7, 5, 11, 11, 11]
